In [1]:
# ============================================================
# MODEL COMPARISON - WATER QUALITY PREDICTION
# Predict:
#   1. pH
#   2. Dissolved Oxygen (DO)
#
# Models:
#   - Linear Regression
#   - Random Forest
#   - Gradient Boosting
#
# Validation:
#   Chronological 80/20 time-based split
# ============================================================

import pandas as pd
import numpy as np
import os
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1. LOAD DATASET
# ============================================================

FILE = "SEAN_FQ_Q_2011-2024_KLGO_TA(2).csv"

df = pd.read_csv("C:/Users/mehar/Desktop/AI_Aquatic_Habitat_Assessment (2)/AI_Aquatic_Habitat_Assessment/ml/SEAN_FQ_Q_2011-2024_KLGO_TA.csv"
                ,low_memory=False)
print("\nDataset loaded successfully!")
print("Original rows:", len(df))


# ============================================================
# 2. SELECT REQUIRED COLUMNS
# ============================================================

features = [
    "water_temperature",
    "sp_conductance",
    "turbidity"
]

targets = [
    "ph",
    "do_concentration"
]

required_columns = features + targets + ["timestamp_utc"]

df = df[required_columns].copy()


# ============================================================
# 3. CONVERT DATA TO NUMERIC
# ============================================================

for column in features + targets:
    df[column] = pd.to_numeric(df[column], errors="coerce")


# ============================================================
# 4. CONVERT TIMESTAMP
# ============================================================

df["timestamp_utc"] = pd.to_datetime(
    df["timestamp_utc"],
    errors="coerce",
    utc=True
)


# ============================================================
# 5. REMOVE MISSING VALUES
# ============================================================

df = df.dropna(subset=required_columns)

print("Rows after removing missing values:", len(df))


# ============================================================
# 6. PHYSICAL VALIDITY FILTER
# ============================================================

df = df[
    (df["water_temperature"] >= 0) &
    (df["water_temperature"] <= 40) &

    (df["sp_conductance"] >= 0) &

    (df["turbidity"] >= 0) &

    (df["ph"] >= 0) &
    (df["ph"] <= 14) &

    (df["do_concentration"] >= 0)
]

print("Rows after physical filtering:", len(df))


# ============================================================
# 7. SORT CHRONOLOGICALLY
# ============================================================

df = df.sort_values("timestamp_utc").reset_index(drop=True)


# ============================================================
# 8. TIME-BASED 80/20 SPLIT
# ============================================================

split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

print("\n================ TIME SPLIT ================")

print(
    "Training period:",
    train_df["timestamp_utc"].min(),
    "to",
    train_df["timestamp_utc"].max()
)

print(
    "Testing period:",
    test_df["timestamp_utc"].min(),
    "to",
    test_df["timestamp_utc"].max()
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))


# ============================================================
# 9. PREPARE FEATURES
# ============================================================

X_train = train_df[features]
X_test = test_df[features]


# ============================================================
# 10. DEFINE MODELS
# ============================================================

models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        random_state=42
    )
}


# ============================================================
# 11. FUNCTION TO TRAIN AND EVALUATE
# ============================================================

def evaluate_models(target):

    print("\n\n================================================")
    print("TARGET:", target)
    print("================================================")

    y_train = train_df[target]
    y_test = test_df[target]

    results = []

    trained_models = {}

    for name, model in models.items():

        print("\nTraining:", name)

        # Train
        model.fit(X_train, y_train)

        # Predict
        predictions = model.predict(X_test)

        # Metrics
        mae = mean_absolute_error(
            y_test,
            predictions
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                predictions
            )
        )

        r2 = r2_score(
            y_test,
            predictions
        )

        results.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        trained_models[name] = model

        print("MAE :", round(mae, 4))
        print("RMSE:", round(rmse, 4))
        print("R²  :", round(r2, 4))


    # ========================================================
    # CREATE RESULTS TABLE
    # ========================================================

    results_df = pd.DataFrame(results)

    print("\n--------------- COMPARISON ---------------")

    print(
        results_df.to_string(
            index=False
        )
    )


    # ========================================================
    # SELECT BEST MODEL
    #
    # We primarily use MAE and RMSE.
    # Lower = better.
    # R² is also considered.
    # ========================================================

    best_model_name = (
        results_df
        .sort_values(
            by=["MAE", "RMSE", "R2"],
            ascending=[True, True, False]
        )
        .iloc[0]["Model"]
    )

    best_model = trained_models[best_model_name]

    print("\nBEST MODEL:", best_model_name)


    return results_df, best_model_name, best_model


# ============================================================
# 12. TRAIN + COMPARE pH MODELS
# ============================================================

ph_results, best_ph_name, best_ph_model = evaluate_models("ph")


# ============================================================
# 13. TRAIN + COMPARE DO MODELS
# ============================================================

do_results, best_do_name, best_do_model = evaluate_models(
    "do_concentration"
)


# ============================================================
# 14. CREATE MODELS FOLDER
# ============================================================

os.makedirs("models", exist_ok=True)


# ============================================================
# 15. SAVE BEST MODELS
# ============================================================

joblib.dump(
    best_ph_model,
    "models/best_ph_model.pkl"
)

joblib.dump(
    best_do_model,
    "models/best_do_model.pkl"
)


# ============================================================
# 16. SAVE COMPARISON RESULTS
# ============================================================

ph_results.to_csv(
    "models/ph_model_comparison.csv",
    index=False
)

do_results.to_csv(
    "models/do_model_comparison.csv",
    index=False
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n\n================================================")
print("FINAL SUMMARY")
print("================================================")

print("Best pH model:", best_ph_name)
print("Best DO model:", best_do_name)

print("\nSaved files:")

print("models/best_ph_model.pkl")
print("models/best_do_model.pkl")
print("models/ph_model_comparison.csv")
print("models/do_model_comparison.csv")

print("\nModel comparison completed successfully!")


Dataset loaded successfully!
Original rows: 61552
Rows after removing missing values: 56501
Rows after physical filtering: 56276

================ TIME SPLIT ================
Training period: 2011-04-25 23:00:51+00:00 to 2022-07-31 17:00:00+00:00
Testing period: 2022-07-31 18:00:00+00:00 to 2024-10-24 21:00:00+00:00
Training rows: 45020
Testing rows: 11256


TARGET: ph

Training: Linear Regression
MAE : 0.0747
RMSE: 0.0937
R²  : 0.5722

Training: Random Forest
MAE : 0.0819
RMSE: 0.1073
R²  : 0.4385

Training: Gradient Boosting
MAE : 0.0683
RMSE: 0.0896
R²  : 0.6087

--------------- COMPARISON ---------------
            Model      MAE     RMSE       R2
Linear Regression 0.074724 0.093678 0.572188
    Random Forest 0.081872 0.107319 0.438517
Gradient Boosting 0.068283 0.089596 0.608657

BEST MODEL: Gradient Boosting


TARGET: do_concentration

Training: Linear Regression
MAE : 0.3638
RMSE: 0.4662
R²  : 0.2229

Training: Random Forest
MAE : 0.3754
RMSE: 0.4826
R²  : 0.1671

Training: Gr